In [3]:
model_name = "llama3.2:1B"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [4]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = ChatOllama(model=model_name)

In [5]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="bge-m3")

In [7]:
from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection_name = "machine_learning_docs"
collection = client.get_or_create_collection(name=collection_name)

# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embedding_model
)

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)


# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = prompt | model | StrOutputParser()

In [10]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query)

print(docs)

[]


In [11]:
#Formatiere Dokumente erstellen und als Input übergeben
formatted_docs = format_docs(docs)
chain.invoke({"docs": formatted_docs})

"However, I don't see any retrieved documents. Please provide the URLs or text of the documents you'd like me to summarize, and I'll do my best to identify the main themes and provide a concise summary."

In [12]:
# Simple stream the chain output
for chunk in chain.stream({"docs": formatted_docs}):
    print(chunk, end="", flush=True)

However, I don't see any retrieved documents. Could you please provide the text of the documents you'd like me to summarize? I'll do my best to identify the main themes and summarize them for you.

In [13]:
# More complex async event streaming
async for event in chain.astream_events({"docs": formatted_docs}, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

However, I don't see any retrieved docs. Please provide the document IDs or URLs of the docs you'd like me to summarize, and I'll do my best to provide an overview of the main themes.

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [19]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = PromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever()

# ADD HERE YOUR CODE
qa_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

In [20]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001E3A73B2750>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}")
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.6'}}, model='llama3.2:1B')
| StrOutputParser()

In [21]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(question)

'Supervised learning is a type of machine learning where an algorithm is trained on labeled data, where the correct output is already known. The algorithm learns to make predictions or classify inputs based on the labeled data, with the goal of minimizing the error between the predicted output and the actual output. This process involves a teacher providing the correct output for a specific input, and the algorithm adjusts its predictions accordingly.'

In [22]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where an algorithm is trained on labeled data to learn the patterns and relationships between inputs and outputs. The labeled data is used to provide the algorithm with examples of correct and incorrect predictions, allowing it to learn and improve its performance over time. By training on labeled data, the algorithm learns to make predictions or take actions based on the patterns it has learned from the data.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [23]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [24]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

C:\Users\lucas\AppData\Local\Temp\ipykernel_8908\2064999211.py:2: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
c:\Users\lucas\Desktop\STUDIUM\DIGITAL ENGINEERING\5. Semester\Technische Informatik\build-your-own-chatbot-stud\.venv\Lib\site-packages\pydantic\main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memor

In [25]:
qa_chain = ConversationalRetrievalChain.from_llm(
    model, retriever=retriever, memory=memory, verbose=False
)

In [26]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where an algorithm is trained on labeled data, meaning the data is already classified or tagged with the correct output. The goal is to learn a mapping between inputs (features) and outputs (labels), so that the algorithm can make predictions or take actions on new, unseen data.

In supervised learning, the training data is used to train the model, and the model is then used to make predictions or take actions on new data. The model learns from the labeled data and improves its performance over time, with the goal of predicting the output (label) for a new input (feature).

Supervised learning is contrasted with unsupervised learning, where the algorithm is not trained on labeled data, and is instead used to discover patterns or structure in the data on its own.

In [27]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Here is the rephrased follow-up question:

Which machine learning algorithms can be used in supervised learning, where an algorithm is trained on labeled data?In supervised learning, where an algorithm is trained on labeled data, the following machine learning algorithms are commonly used:

1. Logistic Regression: A linear model used for binary classification problems.
2. Decision Trees: A simple tree-based model used for classification and regression tasks.
3. Random Forest: An ensemble method that combines multiple decision trees for classification and regression tasks.
4. Support Vector Machines (SVMs): A linear or non-linear model used for classification and regression tasks.
5. Neural Networks: A deep learning model that can learn complex patterns in data, used for classification, regression, and other tasks.
6. K-Nearest Neighbors (KNN): A simple model that uses the k most similar data points to make predictions.
7. Gradient Boosting: An ensemble method that combines multiple wea